# Python Bridge for 11a

## A Short Code Refresher

Use this optional notebook alongside 11a to practice working with monthly dates, smoothing a Series, and inspecting a decomposition result. Allow about 20 minutes. All 36 observations are constructed below, so the notebook runs without a data download.

The lecture supplies the interpretation of trend-cycle, seasonality, and remainder. This companion focuses on the Python steps connecting a dated table to those component Series.

## References

The pandas [time-series guide](https://pandas.pydata.org/docs/user_guide/timeseries.html) covers date conversion, frequency, and date-based selection. The statsmodels references for [seasonal_decompose](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html) and [STL](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.STL.html) explain their returned components and different smoothing settings.

## Turn Date Strings Into an Index

The teaching series has a linear increase and a fixed annual pattern whose twelve effects sum to zero. Its values are illustrative measurement units. We know its construction so we can concentrate on the returned Python objects rather than infer the cause of an empirical pattern.

Create a table with date strings, then parse those strings and move the dates into the index. `format="%Y-%m"` means a four-digit year followed by a two-digit month.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.seasonal import STL, seasonal_decompose

dates = pd.date_range("2020-01-01", periods=36, freq="MS")
seasonal_pattern = np.array([-3, -2, 0, 1, 2, 3, 4, 3, 1, -1, -3, -5])
values = 80 + 0.4 * np.arange(36) + np.tile(seasonal_pattern, 3)
raw = pd.DataFrame({"Month": dates.strftime("%Y-%m"), "Value": values})
raw["Month"] = pd.to_datetime(raw["Month"], format="%Y-%m")
series = raw.set_index("Month")["Value"].sort_index()

display(raw.head())
print("Index type:", type(series.index).__name__)

The observations now have a `DatetimeIndex`. The index labels tell pandas which month each value describes; sorting puts the observations in chronological order.

## Check the Calendar Before Using Row Offsets

Check uniqueness and the expected sequence of dates. Then declare month-start frequency with `.asfreq("MS")`. A frequency is a spacing convention; it does not prove that a source contained every expected observation.

In [ ]:
assert series.index.is_unique
expected = pd.date_range(series.index.min(), series.index.max(), freq="MS")
assert series.index.equals(expected)
series = series.asfreq("MS")

print("Frequency:", series.index.freqstr)
print("Missing values:", int(series.isna().sum()))
display(series.loc["2021-01":"2021-04"])

The four displayed values belong to January through April 2021. The date slice includes both endpoints. With this complete calendar, twelve rows span one year.

## Inspect a Rolling Average

Calculate a centered three-observation mean. `center=True` labels the average at the middle observation, so the first and last values are unavailable without neighboring observations beyond the record.

In [ ]:
centered = series.rolling(window=3, center=True).mean()
smoothing = pd.DataFrame({"Observed": series, "Centered mean": centered})
display(smoothing.head())
display(smoothing.tail(2))

For February 2020, the mean uses January, February, and March. This is a retrospective smoother: at the end of February, March's observation would still be in the future. The lecture's later forecasting work needs an explicit information boundary.

## Read the Classical Decomposition Result

`seasonal_decompose` returns a result object with a Series for each component. Use the same index to display those Series beside the observations. `period=12` specifies twelve monthly observations per annual cycle.

In [ ]:
classical = seasonal_decompose(series, model="additive", period=12)
components = pd.DataFrame(
    {
        "Observed": series,
        "Trend-cycle": classical.trend,
        "Seasonal": classical.seasonal,
        "Remainder": classical.resid,
    }
)
display(components.iloc[4:9].round(3))
print("Unavailable values by column:")
print(components.isna().sum())

The default centered trend estimate leaves six unavailable values at each end. The remainder is also unavailable there because calculating it requires the trend-cycle. Keep those dates visible until you know which calculation needs complete component rows.

## Reconstruct and Seasonally Adjust

Select rows with all components available, then check reconstruction. For an additive decomposition, removing seasonality alone retains the trend-cycle and remainder.

In [ ]:
complete = components.dropna()
rebuilt = complete["Trend-cycle"] + complete["Seasonal"] + complete["Remainder"]
assert np.allclose(rebuilt, complete["Observed"])
adjusted = series - classical.seasonal

display(pd.DataFrame({"Observed": series, "Seasonally adjusted": adjusted}).head())
print("Rows with all classical components:", len(complete))

There are 24 complete component rows. Reconstruction is an accounting check of the returned components, not evidence that the decomposition identifies causes or forecasts well. Seasonal adjustment retains the measurement level; subtracting an earlier year's observation would instead produce a change.

## Fit STL and Inspect Its Components

`STL(...)` constructs a model and `.fit()` returns its fitted decomposition. Here `period=12` sets the cycle; `seasonal=13` is an odd smoother length across successive occurrences of the same calendar month. The two arguments have different jobs.

In [ ]:
stl = STL(series, period=12, seasonal=13, robust=True).fit()
stl_components = pd.DataFrame(
    {"Trend-cycle": stl.trend, "Seasonal": stl.seasonal, "Remainder": stl.resid}
)
assert np.allclose(stl_components.sum(axis=1), series)
display(stl_components.head().round(3))

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True, layout="constrained")
series.plot(ax=axes[0], label="Observed")
stl.trend.plot(ax=axes[0], label="STL trend-cycle")
stl.seasonal.plot(ax=axes[1], label="STL seasonal component")
for ax in axes:
    ax.set_ylabel("Measurement units")
    ax.legend()
axes[0].set_title("Components of the Constructed Monthly Series")
plt.show()

STL returns a trend estimate at the endpoints as well. Endpoint availability does not make that estimate a forecast. The simple planted pattern makes both methods easy to inspect; empirical data need their own checks.

## Try One Small Modification

Remove June 2020 and restore month-start frequency. Predict whether pandas will discard the absent date or insert it with a missing value.

In [ ]:
with_gap = series.drop(pd.Timestamp("2020-06-01"))
regular_calendar = with_gap.asfreq("MS")
display(regular_calendar.loc["2020-05":"2020-07"])

June returns as `NaN`. `.asfreq()` creates the expected calendar; it does not invent the missing measurement. Resolve such gaps deliberately before passing a series to a decomposition routine.

## Ready for 11a

Return to the lecture when you can parse and check a date index, select a date range, explain a rolling average's missing endpoints, and extract and recombine decomposition components. Those operations also support the decomposition concepts introduced in 10b.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).